In [8]:
import pandas as pd
import numpy as np
import datetime as dt
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil

In [9]:
# Read the CSV and extract column names as a list
cols = pd.read_csv("../columns.csv")["Column Names"].tolist()
sensors = pd.read_csv("../../Planning/Sensor_Mapping.csv")
print(cols)  # Display the list of column names

daily_path = '../planning_data/Annual Reports/pre_covid_daily_entries.csv'
database_path = '../planning_data/database/database_cleaned.parquet'
daily_path_database = '../planning_data/database/daily.parquet'
sensors

['From', 'To Time', 'Z01_T4-ChurchSt_RevDoor_IN', 'Z01_T4-ChurchSt_RevDoor_OUT', 'Z01_T4-ChurchSt_SwingDoor_IN', 'Z01_T4-ChurchSt_SwingDoor_OUT', 'Z02_T4-LibertySt_EastEsc46_IN', 'Z02_T4-LibertySt_EastEsc46_OUT', 'Z02_T4-LibertySt_WestEsc45_IN', 'Z02_T4-LibertySt_WestEsc45_OUT', 'Z02_T4-LibertySt_Stair_IN', 'Z02_T4-LibertySt_Stair_OUT', 'Z02_T4-LibertySt_Doors_IN', 'Z02_T4-LibertySt_Doors_OUT', 'Z03_T4-C1-StairEsc_StairElev_IN', 'Z03_T4-C1-StairEsc_StairElev_OUT', 'Z03_T4-C1-StairEsc_EastEsc46_IN', 'Z03_T4-C1-StairEsc_EastEsc46_OUT', 'Z03_T4-C1-StairEsc_WestEsc45_IN', 'Z03_T4-C1-StairEsc_WestEsc45_OUT', 'Z04_T4-SoConc-Entry_AllDoors_IN', 'Z04_T4-SoConc-Entry_AllDoors_OUT', 'Z05_WestProj-Street_NorthDoors_IN', 'Z05_WestProj-Street_NorthDoors_OUT', 'Z05_WestProj-Street_SouthDoors_IN', 'Z05_WestProj-Street_SouthDoors_OUT', 'Z06_WestProj-C1-StairEsc_NorthStair_IN', 'Z06_WestProj-C1-StairEsc_NorthStair_OUT', 'Z06_WestProj-C1-StairEsc_NorthEsc36_IN', 'Z06_WestProj-C1-StairEsc_NorthEsc36_OUT'

,Sensor Number,Sensor Names,Weekly Avg Mapping,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Zone Description - Retail Report,Zone Number
0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2,Z01_T4-ChurchSt_RevDoor_OUT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,3,Z01_T4-ChurchSt_SwingDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,4,Z01_T4-ChurchSt_SwingDoor_OUT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,5,Z02_T4-LibertySt_EastEsc46_IN,4.0,IN,Entries,Street Level,4 WTC Transit Lobby,NaN,NaN,2
...,...,...,...,...,...,...,...,...,...,...
181,182,Z31_T3Elev-C2_INTOELEV_OUT,NaN,OUT,NaN,NaN,NaN,NaN,NaN,31
182,183,Z29_C2_SWOculus_HubElev_OUTOFELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
183,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
184,185,Z28_1Train-C1-SConc_AllDoors_IN,28.0,IN,Entries,NaN,NaN,NaN,NaN,28


In [10]:
df_database = (
    pd.read_parquet(database_path, engine="pyarrow")
    .drop("To Time", axis=1, errors="ignore")  # Avoid KeyError
    .assign(From=lambda df: df["From"].dt.date)  # Convert datetime to date
    .melt(id_vars=["From"], var_name="Location", value_name="Count")  # Reshape data
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")
)

df_database

,From,Location,Count,Sensor Number,Sensor Names,Weekly Avg Mapping,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Zone Description - Retail Report,Zone Number
0,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,1.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99797179,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797180,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,3.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797181,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797182,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29


In [11]:
# df_database = df_database_test
df_database

,From,Location,Count,Sensor Number,Sensor Names,Weekly Avg Mapping,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Zone Description - Retail Report,Zone Number
0,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,1.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99797179,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797180,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,3.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797181,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
99797182,2025-03-31,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29


In [12]:
# df_test['From'] = pd.to_datetime(df_test['From']).dt.date
# df_filtered = df_test[df_test['From'] == pd.to_datetime("2020-02-24").date()]
cols = ["From","Location","Count","Hub In/Out"]

dailys = df_database[cols]
# dailys
dailys = dailys[dailys["Hub In/Out"]=="IN"]
dailys

,From,Location,Count,Hub In/Out
2146176,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,0.0,IN
2146177,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,0.0,IN
2146178,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,0.0,IN
2146179,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,0.0,IN
2146180,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,0.0,IN
...,...,...,...,...
98187547,2025-03-31,Z31_T3Elev-C2_OUTOFELEV_IN,0.0,IN
98187548,2025-03-31,Z31_T3Elev-C2_OUTOFELEV_IN,0.0,IN
98187549,2025-03-31,Z31_T3Elev-C2_OUTOFELEV_IN,0.0,IN
98187550,2025-03-31,Z31_T3Elev-C2_OUTOFELEV_IN,0.0,IN


In [13]:
dailys = dailys[cols]

# Ensure "Count" is numeric (convert non-numeric values to NaN)
dailys["Count"] = pd.to_numeric(dailys["Count"], errors="coerce")

# Group by date and sum counts
dailys = dailys.groupby(["From","Location"], as_index=False).agg({"Count": "sum"})

dailys

,From,Location,Count
0,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,10870.0
1,2020-02-24,Z02_T4-LibertySt_Stair_IN,4087.0
2,2020-02-24,Z02_T4-LibertySt_WestEsc45_IN,1554.0
3,2020-02-24,Z04_T4-SoConc-Entry_AllDoors_IN,6088.0
4,2020-02-24,Z06_WestProj-C1-StairEsc_Elev_OUTOFELEV,813.0
...,...,...,...
119227,2025-03-31,Z28_T3Elev-C1_OUTOFELEV_IN,242.0
119228,2025-03-31,Z29_1Train-C2-SWOculus_AllDoors_IN,2409.0
119229,2025-03-31,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,0.0
119230,2025-03-31,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,0.0


In [14]:
df = pd.read_csv(daily_path)\
    .drop(["To Time"], axis = 1)\
    .melt(id_vars = "From",var_name= "Location",value_name="Count")\
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")

df

cols = ["From","Location","Count","Hub In/Out"]

df = df[cols]
# dailys
df = df[df["Hub In/Out"]=="IN"]\
    .drop("Hub In/Out",axis = 1)\
    .dropna()\
    .reset_index(drop = True)
df

,From,Location,Count
0,1/1/2019,Z02_T4-LibertySt_EastEsc46_IN,9870.0
1,1/2/2019,Z02_T4-LibertySt_EastEsc46_IN,5697.0
2,1/3/2019,Z02_T4-LibertySt_EastEsc46_IN,6187.0
3,1/4/2019,Z02_T4-LibertySt_EastEsc46_IN,2916.0
4,1/5/2019,Z02_T4-LibertySt_EastEsc46_IN,748.0
...,...,...,...
24752,2/19/2020,Z31_T3Elev-C2_OUTOFELEV_IN,241.0
24753,2/20/2020,Z31_T3Elev-C2_OUTOFELEV_IN,265.0
24754,2/21/2020,Z31_T3Elev-C2_OUTOFELEV_IN,156.0
24755,2/22/2020,Z31_T3Elev-C2_OUTOFELEV_IN,70.0


In [15]:
dailys
daily_database = pd.concat([df,dailys], axis = 0, ignore_index = True)
daily_database

,From,Location,Count
0,1/1/2019,Z02_T4-LibertySt_EastEsc46_IN,9870.0
1,1/2/2019,Z02_T4-LibertySt_EastEsc46_IN,5697.0
2,1/3/2019,Z02_T4-LibertySt_EastEsc46_IN,6187.0
3,1/4/2019,Z02_T4-LibertySt_EastEsc46_IN,2916.0
4,1/5/2019,Z02_T4-LibertySt_EastEsc46_IN,748.0
...,...,...,...
143984,2025-03-31,Z28_T3Elev-C1_OUTOFELEV_IN,242.0
143985,2025-03-31,Z29_1Train-C2-SWOculus_AllDoors_IN,2409.0
143986,2025-03-31,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,0.0
143987,2025-03-31,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,0.0


In [16]:
daily_database['From'] = pd.to_datetime(daily_database['From'])
# daily_database['Year'] = daily_database['From'].dt.year
daily_database


,From,Location,Count
0,2019-01-01,Z02_T4-LibertySt_EastEsc46_IN,9870.0
1,2019-01-02,Z02_T4-LibertySt_EastEsc46_IN,5697.0
2,2019-01-03,Z02_T4-LibertySt_EastEsc46_IN,6187.0
3,2019-01-04,Z02_T4-LibertySt_EastEsc46_IN,2916.0
4,2019-01-05,Z02_T4-LibertySt_EastEsc46_IN,748.0
...,...,...,...
143984,2025-03-31,Z28_T3Elev-C1_OUTOFELEV_IN,242.0
143985,2025-03-31,Z29_1Train-C2-SWOculus_AllDoors_IN,2409.0
143986,2025-03-31,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,0.0
143987,2025-03-31,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,0.0


In [17]:
test = daily_database.groupby("From").sum("Count")
test

,Count
From,
2019-01-01,130535.0
2019-01-02,256212.0
2019-01-03,258912.0
2019-01-04,245564.0
2019-01-05,75039.0
...,...
2025-03-27,165874.0
2025-03-28,127068.0
2025-03-29,105654.0


In [ ]:
daily_database.to_parquet(daily_path_database,engine="pyarrow",index = False)
test_read = pd.read_parquet(daily_path_database)
print(test_read.head())

        From                       Location   Count
0 2019-01-01  Z02_T4-LibertySt_EastEsc46_IN  9870.0
1 2019-01-02  Z02_T4-LibertySt_EastEsc46_IN  5697.0
2 2019-01-03  Z02_T4-LibertySt_EastEsc46_IN  6187.0
3 2019-01-04  Z02_T4-LibertySt_EastEsc46_IN  2916.0
4 2019-01-05  Z02_T4-LibertySt_EastEsc46_IN   748.0


: 